# 🤖 LLM Training with Large Context (40-50M tokens)
## Optimized for Google Colab with GPU

**Instructions:**
1. Go to Runtime → Change runtime type → Select GPU (T4 or better)
2. Run cells in order
3. Upload your training files when prompted

In [ ]:
# Cell 1: Install Dependencies
!pip install -q torch transformers datasets accelerate peft bitsandbytes scipy sentencepiece protobuf
print("✅ Dependencies installed!")

In [ ]:
# Cell 2: Import Libraries
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import Dataset
from google.colab import files
import json
import os
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig
import gc

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Cell 3: Configuration
CONFIG = {
    # Model settings
    "model_name": "mistralai/Mistral-7B-v0.1",  # Options: mistralai/Mistral-7B-v0.1, meta-llama/Llama-2-7b-hf, google/gemma-7b
    "max_seq_length": 8192,  # Increase to 16384 or 32768 if you have A100
    
    # Training hyperparameters
    "batch_size": 1,
    "gradient_accumulation_steps": 32,  # Effective batch size = 32
    "learning_rate": 2e-4,
    "num_epochs": 3,
    "warmup_steps": 100,
    "save_steps": 500,
    "logging_steps": 10,
    
    # Memory optimization
    "use_4bit": True,  # 4-bit quantization
    "use_lora": True,  # LoRA fine-tuning
    
    # Output
    "output_dir": "./llm_trained_model",
}

print("⚙️ Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Cell 4: Upload Training Files
print("📁 Upload your training files (txt, json, jsonl)")
print("Supported formats:")
print("  - .txt: Plain text files")
print("  - .json: JSON with 'text' field or array of objects")
print("  - .jsonl: JSON Lines format")
print("\n⬆️ Click 'Choose Files' below...\n")

uploaded = files.upload()
file_paths = list(uploaded.keys())

print(f"\n✅ Uploaded {len(file_paths)} file(s):")
for fp in file_paths:
    size_mb = os.path.getsize(fp) / (1024 * 1024)
    print(f"  - {fp} ({size_mb:.2f} MB)")

In [ ]:
# Cell 5: Data Preparation
def prepare_dataset(file_paths, tokenizer):
    """Load and prepare dataset from uploaded files"""
    all_texts = []
    
    for file_path in file_paths:
        print(f"📖 Processing {file_path}...")
        
        if file_path.endswith('.txt'):
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
                # Split large texts into chunks
                if len(content) > 100000:
                    chunks = [content[i:i+100000] for i in range(0, len(content), 100000)]
                    all_texts.extend(chunks)
                else:
                    all_texts.append(content)
        
        elif file_path.endswith('.json'):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    for item in data:
                        text = item.get('text', item.get('content', str(item)))
                        all_texts.append(text)
                else:
                    text = data.get('text', data.get('content', str(data)))
                    all_texts.append(text)
        
        elif file_path.endswith('.jsonl'):
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        item = json.loads(line)
                        text = item.get('text', item.get('content', str(item)))
                        all_texts.append(text)
    
    print(f"\n📊 Total samples: {len(all_texts)}")
    
    # Create dataset
    dataset = Dataset.from_dict({"text": all_texts})
    
    # Tokenization
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=CONFIG["max_seq_length"],
            padding="max_length",
        )
    
    print("🔄 Tokenizing dataset...")
    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dataset.column_names,
        desc="Tokenizing"
    )
    
    return tokenized_dataset

print("✅ Data preparation function ready")

In [ ]:
# Cell 6: Load Model and Tokenizer
print("🔧 Loading model and tokenizer...")
print(f"Model: {CONFIG['model_name']}\n")

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=CONFIG["use_4bit"],
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
) if CONFIG["use_4bit"] else None

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("✅ Tokenizer loaded")

# Load model
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
print("✅ Model loaded")

# Apply LoRA
if CONFIG["use_lora"]:
    model = prepare_model_for_kbit_training(model)
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    model = get_peft_model(model, lora_config)
    print("✅ LoRA applied")
    model.print_trainable_parameters()

print("\n🎉 Model setup complete!")

In [ ]:
# Cell 7: Prepare Training Dataset
train_dataset = prepare_dataset(file_paths, tokenizer)
print(f"\n✅ Training dataset ready: {len(train_dataset)} samples")

In [ ]:
# Cell 8: Setup Training
training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    fp16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    weight_decay=0.001,
    report_to="none",
    load_best_model_at_end=False,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("✅ Trainer initialized")

In [ ]:
# Cell 9: Train Model
print("🚀 Starting training...\n")
print("=" * 50)

# Clear cache
gc.collect()
torch.cuda.empty_cache()

# Train
trainer.train()

print("\n" + "=" * 50)
print("✅ Training complete!")

In [ ]:
# Cell 10: Save Model
print("💾 Saving model...")
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"✅ Model saved to {CONFIG['output_dir']}")

# Save to Google Drive (optional)
save_to_drive = input("\nSave to Google Drive? (y/n): ")
if save_to_drive.lower() == 'y':
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r {CONFIG['output_dir']} /content/drive/MyDrive/
    print("✅ Model copied to Google Drive")

In [ ]:
# Cell 11: Test Model
print("🧪 Testing trained model...\n")

test_prompts = [
    "Once upon a time",
    "The future of AI is",
    "In a world where"
]

model.eval()
for prompt in test_prompts:
    print(f"\n📝 Prompt: {prompt}")
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"🤖 Output: {generated_text}")
    print("-" * 80)

In [ ]:
# Cell 12: Download Model (Optional)
print("📥 Download trained model to your computer")
print("\nCreating zip file...")

!zip -r trained_model.zip {CONFIG['output_dir']}

print("\n⬇️ Downloading...")
files.download('trained_model.zip')
print("✅ Download complete!")